# 06 — Classic Interview Coding Problems

Word count, dedup-latest-record, sessionization, nth-highest-per-group, time-series resampling, and point-in-time (`merge_asof`) joins — worked in pandas. Same problems as `../../spark_practice/notebooks/06_structured_streaming_and_interview_problems.ipynb`, so you can compare the two APIs side by side.

In [1]:
import pandas as pd
import numpy as np

### Problem 1 — Word count

In [2]:
lines = pd.DataFrame({
    "line": ["the quick brown fox", "the lazy dog sleeps", "the fox jumps over the dog"],
})

word_counts = (
    lines["line"].str.lower().str.split()
    .explode()
    .value_counts()
    .rename_axis("word")
    .reset_index(name="count")
)
word_counts

,word,count
0,the,4
1,fox,2
2,dog,2
3,quick,1
4,brown,1
5,lazy,1
6,sleeps,1
7,jumps,1
8,over,1


### Problem 2 — Deduplicate, keeping the latest record per key

Same problem as notebook 02's `drop_duplicates` example, repeated here for comparison against the Spark `row_number()` version — `sort_values` + `drop_duplicates(keep="last")` is the pandas equivalent of `Window(...).orderBy(...)` + `row_number() == 1`.

In [3]:
updates = pd.DataFrame({
    "user_id": [1, 1, 2, 2],
    "email": ["alice@x.com", "alice@new.com", "bob@x.com", "bob@older.com"],
    "updated_at": pd.to_datetime(["2024-01-01", "2024-03-05", "2024-02-10", "2024-01-20"]),
})

latest_per_user = (
    updates.sort_values("updated_at")
    .drop_duplicates(subset="user_id", keep="last")
    .sort_values("user_id")
    .reset_index(drop=True)
)
latest_per_user

,user_id,email,updated_at
0,1,alice@new.com,2024-03-05
1,2,bob@x.com,2024-02-10


### Problem 3 — Sessionization

"Group a user's events into sessions, where a new session starts after 30+ minutes of inactivity." Same pattern as Spark: compute the gap from the previous event within each user via `.diff()` (pandas' `lag`), flag a new-session boundary past the threshold, then a running sum of the boundary flags gives each row its session id.

In [4]:
events = pd.DataFrame({
    "user_id": ["u1", "u1", "u1", "u1", "u2"],
    "event_time": pd.to_datetime([
        "2024-01-01 10:00:00", "2024-01-01 10:05:00",
        "2024-01-01 10:50:00",   # >30 min gap -> new session
        "2024-01-01 10:55:00", "2024-01-01 09:00:00",
    ]),
}).sort_values(["user_id", "event_time"])

gap = events.groupby("user_id")["event_time"].diff()
is_new_session = (gap > pd.Timedelta(minutes=30)) | gap.isna()
events["session_id"] = is_new_session.groupby(events["user_id"]).cumsum()
events

,user_id,event_time,session_id
0,u1,2024-01-01 10:00:00,1
1,u1,2024-01-01 10:05:00,1
2,u1,2024-01-01 10:50:00,2
3,u1,2024-01-01 10:55:00,2
4,u2,2024-01-01 09:00:00,1


### Problem 4 — Nth highest value per group

"Find the 2nd highest salary in each department." `rank(method="dense")` (not `"first"`) so tied top salaries don't push the real 2nd-highest out of position — identical reasoning to the Spark version's use of `dense_rank()`.

In [5]:
emp_salaries = pd.DataFrame({
    "dept": ["eng", "eng", "eng", "sales", "sales"],
    "name": ["a", "b", "c", "d", "e"],
    "salary": [100, 100, 90, 80, 70],
})

emp_salaries["drank"] = emp_salaries.groupby("dept")["salary"].rank(method="dense", ascending=False)
second_highest = emp_salaries[emp_salaries["drank"] == 2].drop(columns="drank")
second_highest
# eng -> c/90 (the two 100s tie for 1st with dense rank, so 90 correctly ranks 2nd)

,dept,name,salary
2,eng,c,90
4,sales,e,70


### Problem 5 — Time-series resampling

"Given raw timestamped events, compute daily totals." `.resample(rule)` on a `DatetimeIndex` buckets rows into fixed time intervals and aggregates — the pandas equivalent of Spark Structured Streaming's `window()` function from the Spark notebooks, minus the streaming/watermarking machinery (this is a batch-only operation).

In [6]:
txns = pd.DataFrame({
    "ts": pd.to_datetime([
        "2024-01-01 08:00", "2024-01-01 15:00", "2024-01-02 09:30",
        "2024-01-03 12:00", "2024-01-03 12:45",
    ]),
    "amount": [10, 20, 30, 5, 15],
}).set_index("ts")

daily_totals = txns.resample("D").agg(total=("amount", "sum"), n=("amount", "count"))
daily_totals

,total,n
ts,,
2024-01-01,30,2
2024-01-02,30,1
2024-01-03,20,2


### Problem 6 — Point-in-time join: `merge_asof`

"For each trade, attach the most recent price quote at or before the trade's timestamp" — an *inexact*, nearest-match-backward join on a sorted key, common in financial/event data and awkward to express as a normal equi-join. `pd.merge_asof` is built exactly for this (both frames must be sorted by the join key).

In [7]:
trades = pd.DataFrame({
    "trade_time": pd.to_datetime(["2024-01-01 09:00:01", "2024-01-01 09:00:04", "2024-01-01 09:00:08"]),
    "ticker": ["AAA"] * 3,
})
quotes = pd.DataFrame({
    "quote_time": pd.to_datetime(["2024-01-01 09:00:00", "2024-01-01 09:00:03", "2024-01-01 09:00:06"]),
    "price": [100.0, 100.5, 101.2],
})

pd.merge_asof(trades.sort_values("trade_time"), quotes.sort_values("quote_time"),
              left_on="trade_time", right_on="quote_time", direction="backward")

,trade_time,ticker,quote_time,price
0,2024-01-01 09:00:01,AAA,2024-01-01 09:00:00,100.0
1,2024-01-01 09:00:04,AAA,2024-01-01 09:00:03,100.5
2,2024-01-01 09:00:08,AAA,2024-01-01 09:00:06,101.2


## Final interview checklist

- Can you explain why `.loc` slicing is end-inclusive and `.iloc` isn't?
- Can you write top-N-per-group / dedup-latest-record from memory with `groupby(...).rank()` or `sort_values` + `drop_duplicates`?
- Can you explain `agg` vs `transform` vs `filter` and give an example of each?
- Can you rank `iterrows`/`itertuples`/`.apply()`/vectorized by speed and explain why?
- Can you say when you'd stop using pandas and reach for Spark/Polars/Dask instead?

**Related practice in this repo:** `../../spark_practice/` works through the same problems (joins, window functions, dedup, sessionization) in PySpark, and `../../sql_postgres_practice/` works through several of them in raw SQL — interviewers often ask you to translate a solution between all three.